# DecodeLabs Project 2: Data Classification Using AI
### Professional Portfolio & Case Study: Iris Species Classification with K-Nearest Neighbors (KNN)

This notebook demonstrates the complete end-to-end Machine Learning pipeline developed for **DecodeLabs Project 2 (Predictive Phase)**. We follow a modular architecture consisting of the following stages:
1. **Input**: Data Loading & Feature Scaling
2. **Process**: Train-Test Splitting & Hyperparameter Tuning (Elbow Method)
3. **Output**: Performance Validation (Classification Report & Confusion Matrix Heatmap)

## 1. Setup and Environment Configuration

To maintain a professional standard, we import our customized modules directly from the `src` directory. This keeps the experimental code in this notebook cleanly separated from production logic.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Inject the project root folder into path so we can load modular scripts
sys.path.append(os.path.abspath('..'))

# Import custom modular functions from our codebase
from src.data_loader import load_iris_data, split_dataset, scale_features
from src.model import IrisKNNClassifier
from src.tuner import evaluate_k_values, find_optimal_k, plot_elbow_curve
from src.metrics import compute_classification_metrics, plot_confusion_matrix

# Set visual defaults
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries loaded successfully.")

## 2. Exploratory Data Analysis (EDA)

Let's load the raw Iris benchmark data. According to Slide 8, the Iris dataset consists of:
- **Samples**: 150 (Balanced, 50 per class)
- **Classes**: 3 (Setosa, Versicolor, Virginica)
- **Dimensions**: 4 features (Sepal Length, Sepal Width, Petal Length, Petal Width)

In [ ]:
# Load raw features and labels
X, y, target_names = load_iris_data()
print(f"Feature matrix shape: {X.shape}")
print(f"Target labels length: {len(y)}")
print(f"Species: {target_names}")

# View first few rows
df_full = X.copy()
df_full['species_label'] = y
df_full['species_name'] = y.map(lambda idx: target_names[idx])
df_full.head()

### Visualizing Feature Distribution & Separation
A pairplot helps us inspect clustering behavior and linear separability among the species.

In [ ]:
sns.pairplot(df_full.drop(columns=['species_label']), hue='species_name', palette='Set2')
plt.suptitle('Iris Benchmark: Sepal & Petal Dimensions by Species', y=1.02, fontsize=14, fontweight='bold')
plt.show()

Notice that **Setosa** is completely separable from the other two species across almost all feature combinations, whereas **Versicolor** and **Virginica** share a slight overlap, making local proximity-based algorithms like KNN highly suitable for this task.

## 3. Structural Integrity: Train-Test Splitting

Following Slide 10 guidelines, we split the data into 80% training set (for pattern recognition) and 20% testing set (for validation). We shuffle the samples to eliminate order bias.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(X, y, test_size=0.2, random_state=42)
print(f"Training set features: {X_train.shape} | Labels: {y_train.shape}")
print(f"Testing set features:  {X_test.shape}  | Labels: {y_test.shape}")

# Confirm class balance is maintained in test set via stratification
print("\nTest Set Class Counts:")
for label_idx, count in y_test.value_counts().items():
    print(f" - {target_names[label_idx].capitalize()}: {count} samples")

## 4. The Gatekeeper Rule: Scaling

As explained in Slide 9, raw features can have different magnitude ranges, introducing spatial distance bias. We apply `StandardScaler` to normalize dimensions to mean=0 and variance=1.

In [ ]:
# Scale training and testing splits
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

# Create comparison dataframe
print("Raw Features (First 3 samples):")
print(X_train.head(3))
print("\nScaled Features (First 3 samples):")
print(pd.DataFrame(X_train_scaled, columns=X.columns).head(3))

## 5. Tuning the Engine: Choosing "K"

Selecting the optimal number of neighbors ($K$) is critical. Slide 12 explains that $K=1$ leads to noise/overfitting, whereas a very high $K$ leads to generic underfitting. We evaluate $K$ from 1 to 25 using 5-fold cross-validation.

In [ ]:
# Run K parameter search sweep
k_values, error_rates = evaluate_k_values(X_train_scaled, y_train, max_k=25)
optimal_k = find_optimal_k(k_values, error_rates)

print(f"Optimal K selected: {optimal_k}")

# Display values
df_tuning = pd.DataFrame({'K': k_values, 'CV Error Rate': error_rates})
df_tuning.head(10)

### Plotting the Elbow Curve
Let's visualize the relationship between $K$ and validation error rates.

In [ ]:
# Plotting the elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_values, error_rates, color='#1e3d59', linestyle='dashed', marker='o',
         markerfacecolor='#ff6e40', markersize=8, linewidth=2, label='Cross-Validation Error')
plt.plot(optimal_k, error_rates[k_values.index(optimal_k)], marker='o', markersize=14, 
         markeredgecolor='red', markerfacecolor='none', markeredgewidth=3, 
         label=f'Optimal K ({optimal_k})')

plt.title('Tuning the Engine: Choosing "K" (Elbow Plot)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('K Value (Number of Neighbors)', fontsize=12)
plt.ylabel('Error Rate (1 - CV Accuracy)', fontsize=12)
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', fontsize=11)
plt.show()

## 6. Training the Final Model

We now train our K-Nearest Neighbors Classifier wrapper using the optimal $K$ value determined during the tuning phase.

In [ ]:
# Initialize and train classifier
clf = IrisKNNClassifier(n_neighbors=optimal_k)
clf.fit(X_train_scaled, y_train)
print(f"KNN classifier initialized with K={optimal_k} and fit on scaled training set.")

## 7. Performance Evaluation & Diagnostics

To validate model generalizability, we evaluate predictions on the held-out scaled test split.

In [ ]:
# Predict on validation test split
y_pred = clf.predict(X_test_scaled)

# Calculate metrics
metrics = compute_classification_metrics(y_test, y_pred, target_names)

print(f"Overall Classification Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro F1-Score: {metrics['macro_f1']:.4f}")

### Multi-Class Performance Report
Let's check the Precision, Recall, and F1-Scores for each individual species.

In [ ]:
metrics_df = pd.DataFrame(metrics['class_metrics']).T
metrics_df

### Diagnostic Tool: Confusion Matrix
As explained in Slide 15, the Confusion Matrix is crucial to check for Type I (False Alarm) and Type II (Missed Detection) errors.

In [ ]:
# Display Confusion Matrix heatmap
plot_confusion_matrix(y_test, y_pred, target_names)

## 8. Conclusion

In this study, we:
- Successfully resolved features scaling to ensure balanced distance boundaries.
- Found the optimal K neighbors value using validation curves.
- Obtained stellar performance metrics on the test data.

This completes the milestones for **DecodeLabs Project 2: Data Classification Using AI**.